In [0]:
# Notebook: 01_generate_customers
# Purpose: Simulate realistic customer master data for ZapFlow platform
# Author: Vishnu
# Layer: Simulation

import random
from datetime import datetime, timedelta
from pyspark.sql import SparkSession
from pyspark.sql.types import *
import pyspark.sql.functions as F

# ── Configuration ─────────────────────────────────────────────
NUM_CUSTOMERS = 10000
OUTPUT_PATH = "/Volumes/zapflow_raw/raw_data/raw_files/customers/"
SEED = 42

random.seed(SEED)

# ── Realistic Indian customer data ────────────────────────────
FIRST_NAMES = [
    "Aarav", "Vivaan", "Aditya", "Vihaan", "Arjun",
    "Sai", "Reyansh", "Ayaan", "Krishna", "Ishaan",
    "Priya", "Ananya", "Diya", "Sneha", "Pooja",
    "Kavya", "Meera", "Rhea", "Nisha", "Divya"
]

LAST_NAMES = [
    "Sharma", "Verma", "Patel", "Reddy", "Kumar",
    "Singh", "Nair", "Iyer", "Rao", "Gupta",
    "Joshi", "Mehta", "Chopra", "Malhotra", "Shah"
]

CITIES = [
    "Mumbai", "Delhi", "Bengaluru", "Hyderabad",
    "Chennai", "Pune", "Kolkata", "Ahmedabad"
]

# Quick commerce operates in specific pin codes per city
CITY_PINCODES = {
    "Mumbai":    ["400001", "400050", "400070", "400080", "400092"],
    "Delhi":     ["110001", "110020", "110030", "110045", "110060"],
    "Bengaluru": ["560001", "560030", "560040", "560068", "560095"],
    "Hyderabad": ["500001", "500018", "500034", "500072", "500089"],
    "Chennai":   ["600001", "600020", "600040", "600083", "600096"],
    "Pune":      ["411001", "411014", "411028", "411045", "411057"],
    "Kolkata":   ["700001", "700020", "700040", "700060", "700080"],
    "Ahmedabad": ["380001", "380015", "380028", "380051", "380063"]
}

CUSTOMER_SEGMENTS = ["premium", "regular", "occasional"]
SEGMENT_WEIGHTS   = [0.15, 0.55, 0.30]   # realistic distribution

ACQUISITION_CHANNELS = ["organic", "referral", "paid_ad", "influencer"]
CHANNEL_WEIGHTS       = [0.30, 0.25, 0.35, 0.10]

# ── Helper functions ───────────────────────────────────────────
def random_date(start_year=2022, end_year=2024):
    start = datetime(start_year, 1, 1)
    end   = datetime(end_year, 12, 31)
    return (start + timedelta(days=random.randint(0, (end - start).days))).date()

def weighted_choice(options, weights):
    return random.choices(options, weights=weights, k=1)[0]

def introduce_data_quality_issues(record, customer_id):
    """
    Intentionally corrupt ~5% of records to simulate real-world dirty data.
    This lets us demonstrate data quality handling in Silver layer.
    """
    issue_type = None

    if customer_id % 50 == 0:          # 2% — missing email
        record["email"] = None
        issue_type = "missing_email"

    elif customer_id % 75 == 0:        # ~1.3% — invalid phone
        record["phone"] = "0000000000"
        issue_type = "invalid_phone"

    elif customer_id % 100 == 0:       # 1% — future registration date
        record["registration_date"] = str(datetime(2099, 1, 1).date())
        issue_type = "future_date"

    record["_data_quality_issue"] = issue_type
    return record

# ── Generate records ───────────────────────────────────────────
def generate_customers(n):
    records = []

    for i in range(1, n + 1):
        customer_id = f"CUST_{i:06d}"
        first_name  = random.choice(FIRST_NAMES)
        last_name   = random.choice(LAST_NAMES)
        city        = random.choice(CITIES)
        segment     = weighted_choice(CUSTOMER_SEGMENTS, SEGMENT_WEIGHTS)

        # Premium customers registered earlier — realistic behavior
        if segment == "premium":
            reg_date = random_date(2022, 2023)
        elif segment == "regular":
            reg_date = random_date(2022, 2024)
        else:
            reg_date = random_date(2023, 2024)

        record = {
            "customer_id":          customer_id,
            "first_name":           first_name,
            "last_name":            last_name,
            "email":                f"{first_name.lower()}.{last_name.lower()}{i}@gmail.com",
            "phone":                f"9{random.randint(100000000, 999999999)}",
            "city":                 city,
            "pincode":              random.choice(CITY_PINCODES[city]),
            "customer_segment":     segment,
            "acquisition_channel":  weighted_choice(ACQUISITION_CHANNELS, CHANNEL_WEIGHTS),
            "registration_date":    str(reg_date),
            "is_active":            random.choices([True, False], weights=[0.90, 0.10])[0],
            "_data_quality_issue":  None
        }

        # Introduce intentional quality issues
        record = introduce_data_quality_issues(record, i)
        records.append(record)

    return records

# ── Main execution ─────────────────────────────────────────────
print("Generating customer data...")
customer_records = generate_customers(NUM_CUSTOMERS)

# Convert to Spark DataFrame
schema = StructType([
    StructField("customer_id",         StringType(),  False),
    StructField("first_name",          StringType(),  True),
    StructField("last_name",           StringType(),  True),
    StructField("email",               StringType(),  True),
    StructField("phone",               StringType(),  True),
    StructField("city",                StringType(),  True),
    StructField("pincode",             StringType(),  True),
    StructField("customer_segment",    StringType(),  True),
    StructField("acquisition_channel", StringType(),  True),
    StructField("registration_date",   StringType(),  True),
    StructField("is_active",           BooleanType(), True),
    StructField("_data_quality_issue", StringType(),  True),
])

df_customers = spark.createDataFrame(customer_records, schema=schema)

# ── Save as CSV (raw simulation output) ───────────────────────
(df_customers
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv(OUTPUT_PATH))

print(f"✅ Generated {NUM_CUSTOMERS} customer records")
print(f"✅ Saved to {OUTPUT_PATH}")

# ── Quick sanity check ─────────────────────────────────────────
print("\n── Segment distribution ──")
df_customers.groupBy("customer_segment").count().orderBy("count", ascending=False).show()

print("── Data quality issues introduced ──")
df_customers.groupBy("_data_quality_issue").count().orderBy("count", ascending=False).show()

print("── Sample records ──")
df_customers.show(5, truncate=False)